# Витрина МСФО: `df_out_msfo` + расчетная витрина `df_out_rez`

Этот ноутбук — модульный шаблон для построения двух связанных витрин:

1. **`df_out_msfo`** — договорный слой: статические фактические поля + блоки на каждую отчетную дату с остатком, процентами, рейтингом, стадией, резервом МСФО и факторным разложением.
2. **`df_out_rez`** — расчетный слой: CCF, срок до погашения, субпортфель, PD PIT, PDLT, default-age и места для будущих формул LGD / резерва.

## Главный принцип

Все, что зависит от конкретного входного файла, вынесено в **один блок конфигурации**:

- имена DataFrame;
- реальные названия колонок;
- ключи сопоставления;
- правила lookup-таблиц;
- отчетные даты;
- формула `%резервирования МСФО`.

Если структура входных данных поменяется, в большинстве случаев достаточно поменять только конфигурацию.

> **Важно по `%резервирования МСФО`.** По постановке используется формула `ОД+проценты в BYN / Резерв по МСФО`. Она реализована буквально как основной режим. Если фактически требуется классический `Резерв / Экспозиция × 100`, достаточно переключить одну настройку `MSFO_RATE_MODE`.


## 1. Импорты и общие настройки

In [1]:
from __future__ import annotations

import math
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)

DATE_FMT = "%d.%m.%Y"
EPS = 1e-12
BASE_CURRENCY = "BYN"

# Если None — даты автоматически берутся из MONTHLY_SOURCE по колонке report_date.
# Можно задать явно, например:
# REPORT_DATES = ["01.10.2026", "01.11.2026", "01.12.2026"]
REPORT_DATES = None

# Формула из постановки пользователя:
#   (ОД + начисленные проценты + просроченные проценты в BYN) / Резерв по МСФО
# Альтернатива: "reserve_div_exposure_pct" -> Резерв / Экспозиция * 100
MSFO_RATE_MODE = "exposure_div_reserve"

# Для факторного анализа ставка трактуется как процент и делится на 100.
RATE_IS_PERCENT = True

# Если BYN-поля есть прямо в monthly source — берем их.
# Если их нет, считаем валютное значение * FX.
PREFER_DIRECT_BYN = True


## 2. Где лежат входные данные

Впиши здесь **имена DataFrame**, которые будут существовать в памяти перед запуском `run_pipeline()`.

Минимально нужны `base` и `monthly`. Остальные источники можно подключать постепенно.

In [2]:
INPUT_DF_NAMES = {
    # Одна строка на договор / валюту. Здесь статические фактические признаки.
    "base": "df_msfo_base",

    # Длинный помесячный источник:
    # ключ договора + Отчетная дата + остаток/проценты/рейтинг/стадия/зона.
    "monthly": "df_msfo_monthly",

    # Отдельная таблица фактического/прогнозного резерва МСФО по договору и дате.
    "reserve": "df_msfo_reserve",

    # Курсы. Поддерживается wide-формат как у старого df_rates и long-формат.
    "fx": "df_rates",

    # Lookup-таблицы для df_out_rez.
    "ccf": "df_ccf",
    "macro": "df_macro",
    "pd_pit": "df_pd_pit",
    "pdlt": "df_pdlt",
}


def get_named_df(name: Optional[str], required: bool = False) -> Optional[pd.DataFrame]:
    if not name:
        if required:
            raise NameError("Не задано имя обязательного DataFrame")
        return None

    obj = globals().get(name)

    if obj is None:
        if required:
            raise NameError(
                f"В памяти нет DataFrame {name!r}. Создай его либо поменяй INPUT_DF_NAMES."
            )
        return None

    if not isinstance(obj, pd.DataFrame):
        raise TypeError(f"{name!r} существует, но это не pandas.DataFrame")

    return obj


### 2.1. Пример чтения файлов

Эту ячейку не обязательно выполнять. Она показывает, где именно ты потом можешь указать свои источники.

In [3]:
# Примеры — раскомментируй и поправь пути под свои файлы.
#
# df_msfo_base = pd.read_excel("data/msfo_fact.xlsx")
# df_msfo_monthly = pd.read_excel("data/msfo_monthly.xlsx")
# df_msfo_reserve = pd.read_excel("data/msfo_reserve.xlsx")
# df_rates = pd.read_excel("data/fx_rates.xlsx")
# df_ccf = pd.read_excel("data/ccf.xlsx")
# df_macro = pd.read_excel("data/macro.xlsx")
# df_pd_pit = pd.read_excel("data/pd_pit.xlsx")
# df_pdlt = pd.read_excel("data/pdlt.xlsx")


## 3. Сопоставление реальных колонок входных файлов

Слева — **логическое имя**, которое использует код. Справа — **реальное название колонки** в твоем файле.

Если колонки пока нет или источник будет подключен позже — поставь `None`.

In [4]:
# -----------------------------------------------------------------------------
# 3.1. BASE: статические фактические поля
# -----------------------------------------------------------------------------

BASE_COLS = {
    "balance_type": "Баланс/внебаланс/списанные",
    "contract": "Номер договора",
    "segment_2026": "Сегмент 2026",
    "segment_2027": "Сегмент 2027",
    "unp": "УНП",
    "client": "Наименование клиента",
    "oked": "ОКЭД",
    "industry": "Отрасль",
    "msfo_ind": "Инд оценка МСФО",
    "problem_zone_fact": "Зона проблемности",
    "product": "Продукт",
    "line_type": "Тип линии",
    "multicurrency_flag": "Флаг мультивалютности",
    "currency": "Код валюты",
    "start_date": "Дата начала договора",
    "end_date": "Дата окончания договора",
    "interest_rate_fact": "Фактическая процентная ставка",
    "industry_nbrb": "Отрасль НБ РБ",
    "security": "Обеспеченность",
    "sbl_client": "Клиент СБЛ",
    "maturity_type": "Срочность",
    "rating_fact": "Рейтинг факт",
    "rating_at_contract": "Рейтинг на дату договора",
    "lgd": "LGD",
    "max_dpd_fact": "Максимальное количество дней просрочки факт по ГП/клиенту",
    "default_status_fact": "Статус дефолта фактический",
    "default_date": "Дата дефолта",
    "sanctions": "Санкции",
    "state_subsidies": "Гос субсидии",
    "restra": "Наличие рестры",

    "od_fact_byn": "Остаток факт в BYN",
    "accrued_fact_byn": "Начисленные проценты факт в BYN",
    "overdue_fact_byn": "Просроченные проценты факт в BYN",

    "od_fact_ccy": "Остаток факт в валюте",
    "accrued_fact_ccy": "Начисленные проценты факт в валюте",
    "overdue_fact_ccy": "Просроченные проценты факт в валюте",

    "msfo_rate_fact": "%рез факт по МСФО",
    "risk_group_fact": "Группа риска факт",
    "nsfo_rate_fact": "%рез факт по НСФО",

    # Эти два поля нужны df_out_rez.
    "stage_fact": "Стадия факт",
    "balance_report_date": "Отчетная дата баланса",
}


# -----------------------------------------------------------------------------
# 3.2. MONTHLY: длинный помесячный источник
# -----------------------------------------------------------------------------

MONTHLY_COLS = {
    "balance_type": "Баланс/внебаланс/списанные",
    "contract": "Номер договора",
    "unp": "УНП",
    "currency": "Код валюты",
    "report_date": "Отчетная дата",

    "od_ccy": "Остаток в валюте",
    "accrued_ccy": "Начисленные проценты в валюте",
    "overdue_ccy": "Просроченные проценты в валюте",

    # Если этих колонок нет — поставь None, тогда BYN будет рассчитан через FX.
    "od_byn": "Остаток в BYN",
    "accrued_byn": "Начисленные проценты в BYN",
    "overdue_byn": "Просроченные проценты в BYN",

    "rating": "Рейтинг",
    "problem_zone": "Зона проблемности",
    "stage": "Стадия",
}


# -----------------------------------------------------------------------------
# 3.3. RESERVE: отдельная таблица резерва МСФО
# -----------------------------------------------------------------------------

RESERVE_COLS = {
    "balance_type": "Баланс/внебаланс/списанные",
    "contract": "Номер договора",
    "unp": "УНП",
    "currency": "Код валюты",
    "report_date": "Отчетная дата",
    "reserve": "Резерв по МСФО",
}


# -----------------------------------------------------------------------------
# 3.4. FX: можно использовать long либо wide
# -----------------------------------------------------------------------------

FX_COLS = {
    "date": "Дата",
    "currency": "Валюта",
    "rate": "Курс",
}


# Один договор может быть мультивалютным, поэтому валюта входит в ключ.
# balance_type можно добавить при необходимости.
ENTITY_KEY_FIELDS = [
    "unp",
    "contract",
    "currency",
]

RESERVE_KEY_FIELDS = [
    "unp",
    "contract",
    "currency",
]


## 4. Схемы итоговых DataFrame

In [5]:
MSFO_STATIC_FIELDS = [
    ("balance_type", "Баланс/внебаланс/списанные"),
    ("contract", "Номер договора"),
    ("segment_2026", "Сегмент 2026"),
    ("segment_2027", "Сегмент 2027"),
    ("unp", "УНП"),
    ("client", "Наименование клиента"),
    ("oked", "ОКЭД"),
    ("industry", "Отрасль"),
    ("msfo_ind", "Инд оценка МСФО"),
    ("problem_zone_fact", "Зона проблемности"),
    ("product", "Продукт"),
    ("line_type", "Тип линии"),
    ("multicurrency_flag", "Флаг мультивалютности"),
    ("currency", "Код валюты"),
    ("start_date", "Дата начала договора"),
    ("end_date", "Дата окончания договора"),
    ("interest_rate_fact", "Фактическая процентная ставка"),
    ("industry_nbrb", "Отрасль НБ РБ"),
    ("security", "Обеспеченность"),
    ("sbl_client", "Клиент СБЛ"),
    ("maturity_type", "Срочность"),
    ("rating_fact", "Рейтинг факт"),
    ("rating_at_contract", "Рейтинг на дату договора"),
    ("lgd", "LGD"),
    ("max_dpd_fact", "Максимальное количество дней просрочки факт по ГП/клиенту"),
    ("default_status_fact", "Статус дефолта фактический"),
    ("default_date", "Дата дефолта"),
    ("sanctions", "Санкции"),
    ("state_subsidies", "Гос субсидии"),
    ("restra", "Наличие рестры"),
    ("od_fact_byn", "Остаток факт в BYN"),
    ("accrued_fact_byn", "Начисленные проценты факт в BYN"),
    ("overdue_fact_byn", "Просроченные проценты факт в BYN"),
    ("od_fact_ccy", "Остаток факт в валюте"),
    ("accrued_fact_ccy", "Начисленные проценты факт в валюте"),
    ("overdue_fact_ccy", "Просроченные проценты факт в валюте"),
    ("msfo_rate_fact", "%рез факт по МСФО"),
    ("risk_group_fact", "Группа риска факт"),
    ("nsfo_rate_fact", "%рез факт по НСФО"),
    ("stage_fact", "Стадия факт"),
    ("balance_report_date", "Отчетная дата баланса"),
]

MSFO_PERIOD_BASES = [
    "ОД в валюте",
    "Начисленные проценты в валюте",
    "Просроченные проценты в валюте",
    "ОД+проценты в валюте",
    "ОД в BYN",
    "Начисленные проценты в BYN",
    "Просроченные проценты в BYN",
    "ОД+проценты в BYN",
    "Рейтинг",
    "Изменение рейтинга",
    "Зона проблемности",
    "Стадия",
    "Резерв по МСФО",
    "%резервирования МСФО",
    "Ухудшение качества",
    "Переоценка",
    "Движение портфеля",
]

REZ_STATIC_FIELDS = [
    "Баланс/внебаланс/списанные",
    "Инд",
    "Отчетная дата баланса",
    "Номер договора",
    "ОД+проценты факт в BYN",
    "Стадия факт",
    "%рез факт по МСФО",
]

REZ_PERIOD_BASES = [
    "ОД+проценты в BYN",
    "Стадия",
    "Срок договора",
    "CCF",
    "Инд",
    "Срок до погашения",
    "Субпортфель код",
    "Макронадбавка",
    "PD PIT",
    "Ключ субпортфель_рейтинг",
    "Срок для PDLT max",
    "Срок для PDLT min",
    "PD max",
    "PD min",
    "PDLT_больше года",
    "PDLT",
    "Дата в дефолте",
    "Срок дефолта",
    "LGD ID",
    "Резерв по МСФО",
    "%рез по МСФО",
]


def pcol(base: str, date: str) -> str:
    return f"{base}_{date}"


## 5. Конфигурация lookup-таблиц для `df_out_rez`

Это главное место для источников, которые в постановке пока не были детализированы.

Поддерживаются:

- обычные статические колонки;
- помесячные колонки вида `Стадия_{date}` через шаблон `Стадия_{date}`;
- специальный ключ `__REPORT_DATE__`, который подставляет текущую отчетную дату.


In [6]:
LOOKUP_SPECS = {
    "ccf": {
        "df_name": INPUT_DF_NAMES["ccf"],
        # ПРИМЕР. Если CCF зависит от других признаков — просто поменяй ключи.
        "left_keys": ["Тип линии"],
        "right_keys": ["Тип линии"],
        "value_col": "CCF",
    },

    "macro": {
        "df_name": INPUT_DF_NAMES["macro"],
        # Можно, например, добавить "__REPORT_DATE__" и колонку даты справа.
        "left_keys": ["Субпортфель код_{date}"],
        "right_keys": ["Субпортфель код"],
        "value_col": "Макронадбавка",
    },

    "pd_pit": {
        "df_name": INPUT_DF_NAMES["pd_pit"],
        "left_keys": ["Субпортфель код_{date}"],
        "right_keys": ["Субпортфель код"],
        "value_col": "PD PIT",
    },

    "pdlt_max": {
        "df_name": INPUT_DF_NAMES["pdlt"],
        "left_keys": [
            "Ключ субпортфель_рейтинг_{date}",
            "Срок для PDLT max_{date}",
        ],
        "right_keys": [
            "Ключ субпортфель_рейтинг",
            "Срок",
        ],
        "value_col": "PD",
    },

    "pdlt_min": {
        "df_name": INPUT_DF_NAMES["pdlt"],
        "left_keys": [
            "Ключ субпортфель_рейтинг_{date}",
            "Срок для PDLT min_{date}",
        ],
        "right_keys": [
            "Ключ субпортфель_рейтинг",
            "Срок",
        ],
        "value_col": "PD",
    },
}


## 6. Диагностика и базовые служебные функции

In [7]:
@dataclass
class Diagnostics:
    items: List[Dict[str, Any]] = field(default_factory=list)

    def add(self, issue_type: str, message: str, **kwargs):
        row = {"Тип": issue_type, "Сообщение": message}
        row.update(kwargs)
        self.items.append(row)

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.items)


def to_num(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s, errors="coerce")
    return pd.to_numeric(
        s.astype("string")
         .str.replace("\u00a0", "", regex=False)
         .str.replace(" ", "", regex=False)
         .str.replace(",", ".", regex=False),
        errors="coerce",
    )


def normalize_scalar(value: Any) -> str:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    text = str(value).strip()
    if text.endswith(".0"):
        try:
            text = str(int(float(text)))
        except Exception:
            pass
    return text.lower().replace("ё", "е")




def parse_mixed_date(value: Any):
    """Надежно парсит Timestamp, ISO YYYY-MM-DD и DD.MM.YYYY."""
    if isinstance(value, pd.Timestamp):
        return value
    return pd.to_datetime(value, errors="coerce", format="mixed", dayfirst=True)


def parse_mixed_dates(values):
    """Векторный парсер смешанных форматов дат."""
    return pd.to_datetime(values, errors="coerce", format="mixed", dayfirst=True)

def normalize_date(value: Any) -> pd.Timestamp:
    if value is None or value == "":
        return pd.NaT
    return parse_mixed_date(value).normalize()


def format_date(value: Any) -> str:
    ts = normalize_date(value)
    if pd.isna(ts):
        raise ValueError(f"Некорректная отчетная дата: {value!r}")
    return ts.strftime(DATE_FMT)


def numeric_sum(df: pd.DataFrame, cols: Sequence[str]) -> pd.Series:
    parts = []
    for col in cols:
        if col in df.columns:
            parts.append(to_num(df[col]))
        else:
            parts.append(pd.Series(np.nan, index=df.index, dtype=float))
    return pd.concat(parts, axis=1).sum(axis=1, min_count=1)


def require_mapped_columns(
    df: pd.DataFrame,
    mapping: Dict[str, Optional[str]],
    logical_fields: Sequence[str],
    source_name: str,
):
    missing = []
    for logical in logical_fields:
        real = mapping.get(logical)
        if not real or real not in df.columns:
            missing.append(f"{logical} -> {real!r}")
    if missing:
        raise KeyError(
            f"{source_name}: не хватает обязательных колонок: " + ", ".join(missing)
        )


def composite_key(
    df: pd.DataFrame,
    mapping: Dict[str, Optional[str]],
    logical_fields: Sequence[str],
) -> pd.Series:
    require_mapped_columns(df, mapping, logical_fields, "composite_key")
    normalized = [df[mapping[k]].map(normalize_scalar) for k in logical_fields]
    return pd.Series(list(zip(*normalized)), index=df.index, dtype=object)


def ensure_unique_key_date(
    df: pd.DataFrame,
    key: pd.Series,
    date_series: pd.Series,
    source_name: str,
):
    tmp = pd.DataFrame({"_key": key, "_date": date_series}, index=df.index)
    dup = tmp.duplicated(["_key", "_date"], keep=False)
    if dup.any():
        sample = tmp.loc[dup].head(10)
        raise ValueError(
            f"{source_name}: найдены дубли по ключу договора + отчетной дате. "
            f"Исправь источник либо расширь ENTITY_KEY_FIELDS. Пример:\n{sample}"
        )


def mapped_series(
    df: pd.DataFrame,
    mapping: Dict[str, Optional[str]],
    logical: str,
    diagnostics: Optional[Diagnostics] = None,
    source_name: str = "source",
) -> pd.Series:
    col = mapping.get(logical)
    if col and col in df.columns:
        return df[col].copy()

    if diagnostics is not None:
        diagnostics.add(
            "Нет колонки",
            f"{source_name}: поле {logical!r} не найдено; будет NaN",
            Логическое_поле=logical,
            Колонка=col,
        )
    return pd.Series(np.nan, index=df.index)


## 7. Отчетные даты и FX

In [8]:
def resolve_report_dates(monthly: pd.DataFrame) -> List[str]:
    if REPORT_DATES is not None:
        return sorted(
            {format_date(x) for x in REPORT_DATES},
            key=lambda x: pd.to_datetime(x, format=DATE_FMT),
        )

    report_col = MONTHLY_COLS.get("report_date")
    if not report_col or report_col not in monthly.columns:
        raise KeyError(
            "REPORT_DATES=None, но в monthly не найдена колонка report_date. "
            "Либо задай REPORT_DATES явно, либо поправь MONTHLY_COLS['report_date']."
        )

    dates = parse_mixed_dates(monthly[report_col]).dropna()
    return [d.strftime(DATE_FMT) for d in sorted(dates.dt.normalize().unique())]


def get_fx_map(
    fx: Optional[pd.DataFrame],
    date: str,
    diagnostics: Optional[Diagnostics] = None,
) -> Dict[str, float]:
    result = {normalize_scalar(BASE_CURRENCY): 1.0}

    if fx is None:
        if diagnostics is not None:
            diagnostics.add("Нет FX", f"Нет таблицы FX для даты {date}")
        return result

    date_ts = pd.to_datetime(date, format=DATE_FMT)
    ccol = FX_COLS.get("currency")
    dcol = FX_COLS.get("date")
    rcol = FX_COLS.get("rate")

    # Long format: Дата | Валюта | Курс
    if ccol in fx.columns and dcol in fx.columns and rcol in fx.columns:
        tmp = fx.copy()
        tmp["_date"] = parse_mixed_dates(tmp[dcol]).dt.normalize()
        tmp = tmp[tmp["_date"].eq(date_ts)]
        for _, row in tmp.iterrows():
            result[normalize_scalar(row[ccol])] = pd.to_numeric(row[rcol], errors="coerce")
        return result

    # Wide format: валюты в индексе/колонке, даты в колонках.
    wide = fx.copy()
    if ccol in wide.columns:
        wide = wide.set_index(ccol)

    date_candidates = [
        date,
        date_ts,
        date_ts.strftime("%Y-%m-%d"),
        date_ts.strftime("%d.%m.%Y"),
    ]

    chosen = next((c for c in date_candidates if c in wide.columns), None)
    if chosen is None:
        # пробуем распарсить заголовки
        parsed = {}
        for c in wide.columns:
            ts = parse_mixed_date(c)
            if not pd.isna(ts):
                parsed[ts.normalize()] = c
        chosen = parsed.get(date_ts)

    if chosen is None:
        if diagnostics is not None:
            diagnostics.add("Нет FX даты", f"В FX не найдена дата {date}")
        return result

    for currency, value in wide[chosen].items():
        result[normalize_scalar(currency)] = pd.to_numeric(value, errors="coerce")

    return result


## 8. Generic lookup helper

In [9]:
def _resolve_left_key_series(df: pd.DataFrame, template: str, date: str) -> pd.Series:
    if template == "__REPORT_DATE__":
        return pd.Series(date, index=df.index)

    col = template.format(date=date)
    if col not in df.columns:
        raise KeyError(f"В левой таблице нет lookup-колонки {col!r}")
    return df[col]


def lookup_series(
    left_df: pd.DataFrame,
    date: str,
    spec: Dict[str, Any],
    diagnostics: Diagnostics,
    lookup_name: str,
) -> pd.Series:
    lookup_df = get_named_df(spec.get("df_name"), required=False)
    if lookup_df is None:
        diagnostics.add(
            "Lookup не подключен",
            f"{lookup_name}: DataFrame {spec.get('df_name')!r} не найден; значения будут NaN",
        )
        return pd.Series(np.nan, index=left_df.index, dtype=float)

    left_keys = spec["left_keys"]
    right_keys = spec["right_keys"]
    value_col = spec["value_col"]

    if len(left_keys) != len(right_keys):
        raise ValueError(f"{lookup_name}: left_keys и right_keys разной длины")

    missing_right = [c for c in right_keys + [value_col] if c not in lookup_df.columns]
    if missing_right:
        diagnostics.add(
            "Lookup: нет колонок",
            f"{lookup_name}: отсутствуют {missing_right}; значения будут NaN",
        )
        return pd.Series(np.nan, index=left_df.index, dtype=float)

    try:
        left_parts = [_resolve_left_key_series(left_df, x, date).map(normalize_scalar) for x in left_keys]
    except KeyError as exc:
        diagnostics.add("Lookup: нет левого ключа", f"{lookup_name}: {exc}")
        return pd.Series(np.nan, index=left_df.index, dtype=float)

    right_parts = [lookup_df[x].map(normalize_scalar) for x in right_keys]

    left_key = pd.Series(list(zip(*left_parts)), index=left_df.index)
    right_key = pd.Series(list(zip(*right_parts)), index=lookup_df.index)

    tmp = pd.DataFrame({"_key": right_key, "_value": lookup_df[value_col]})
    duplicate = tmp.duplicated("_key", keep=False)
    if duplicate.any():
        conflicts = (
            tmp.loc[duplicate]
            .groupby("_key")["_value"]
            .nunique(dropna=False)
        )
        conflicts = conflicts[conflicts > 1]
        if len(conflicts):
            raise ValueError(
                f"{lookup_name}: по одному lookup-ключу найдено несколько разных значений. "
                f"Пример ключей: {list(conflicts.index[:5])}"
            )

    mapping = tmp.drop_duplicates("_key", keep="first").set_index("_key")["_value"].to_dict()
    return left_key.map(mapping)


## 9. Формулы `df_out_msfo`

По умолчанию факторные формулы перенесены из предыдущей логики `df_out`, но база заменена на полную экспозицию `ОД + начисленные проценты + просроченные проценты`.

Если для МСФО нужны другие правила, менять нужно только функции в этой секции.

In [10]:
def calculate_msfo_rate(exposure_byn: pd.Series, reserve: pd.Series) -> pd.Series:
    exposure = to_num(exposure_byn)
    reserve_num = to_num(reserve)

    if MSFO_RATE_MODE == "exposure_div_reserve":
        # Буквально по постановке пользователя.
        result = exposure / reserve_num.replace(0, np.nan)
    elif MSFO_RATE_MODE == "reserve_div_exposure_pct":
        result = reserve_num / exposure.replace(0, np.nan) * 100.0
    else:
        raise ValueError(f"Неизвестный MSFO_RATE_MODE={MSFO_RATE_MODE!r}")

    return result.replace([np.inf, -np.inf], np.nan)


def calculate_rating_change(current: pd.Series, rating_at_contract: pd.Series) -> pd.Series:
    current_num = pd.to_numeric(current, errors="coerce")
    contract_num = pd.to_numeric(rating_at_contract, errors="coerce")
    return current_num - contract_num


def calculate_quality_effect(
    current_exposure_byn: pd.Series,
    prev_rate: pd.Series,
    current_rate: pd.Series,
) -> pd.Series:
    base = to_num(current_exposure_byn).fillna(0.0)
    prev = to_num(prev_rate).fillna(0.0)
    curr = to_num(current_rate).fillna(0.0)
    divisor = 100.0 if RATE_IS_PERCENT else 1.0
    return -base * (curr - prev) / divisor


def calculate_revaluation_effect(
    prev_exposure_ccy: pd.Series,
    currency: pd.Series,
    prev_date: str,
    current_date: str,
    prev_rate: pd.Series,
    fx: Optional[pd.DataFrame],
    diagnostics: Diagnostics,
) -> pd.Series:
    prev_fx_map = get_fx_map(fx, prev_date, diagnostics)
    curr_fx_map = get_fx_map(fx, current_date, diagnostics)

    ccy_key = currency.map(normalize_scalar)
    prev_fx = ccy_key.map(prev_fx_map)
    curr_fx = ccy_key.map(curr_fx_map)

    divisor = 100.0 if RATE_IS_PERCENT else 1.0

    return (
        to_num(prev_exposure_ccy).fillna(0.0)
        * (to_num(curr_fx).fillna(0.0) - to_num(prev_fx).fillna(0.0))
        * to_num(prev_rate).fillna(0.0)
        / divisor
    )


def calculate_portfolio_movement(
    prev_exposure_byn: pd.Series,
    current_exposure_byn: pd.Series,
    prev_rate: pd.Series,
    revaluation: pd.Series,
) -> pd.Series:
    divisor = 100.0 if RATE_IS_PERCENT else 1.0
    prev_exp = to_num(prev_exposure_byn).fillna(0.0)
    curr_exp = to_num(current_exposure_byn).fillna(0.0)
    prev_rate_num = to_num(prev_rate).fillna(0.0)

    return -(
        curr_exp * prev_rate_num / divisor
        - prev_exp * prev_rate_num / divisor
    ) - to_num(revaluation).fillna(0.0)


## 10. Построение `df_out_msfo`

In [11]:
def build_df_out_msfo(
    base: pd.DataFrame,
    monthly: pd.DataFrame,
    reserve: Optional[pd.DataFrame] = None,
    fx: Optional[pd.DataFrame] = None,
    diagnostics: Optional[Diagnostics] = None,
) -> Tuple[pd.DataFrame, List[str], Diagnostics]:
    diagnostics = diagnostics or Diagnostics()

    require_mapped_columns(base, BASE_COLS, ENTITY_KEY_FIELDS, "base")
    require_mapped_columns(monthly, MONTHLY_COLS, ENTITY_KEY_FIELDS + ["report_date"], "monthly")

    report_dates = resolve_report_dates(monthly)
    if not report_dates:
        raise ValueError("Список отчетных дат пуст")

    out = pd.DataFrame(index=base.index.copy())

    # -------------------------------------------------------------------------
    # 10.1. Статические поля
    # -------------------------------------------------------------------------
    for logical, output_name in MSFO_STATIC_FIELDS:
        out[output_name] = mapped_series(
            base,
            BASE_COLS,
            logical,
            diagnostics=diagnostics,
            source_name="base",
        ).values

    # Фактические суммы всегда пересчитываем из трех компонентов.
    out["ОД+проценты факт в BYN"] = numeric_sum(
        out,
        [
            "Остаток факт в BYN",
            "Начисленные проценты факт в BYN",
            "Просроченные проценты факт в BYN",
        ],
    )
    out["ОД+проценты факт в валюте"] = numeric_sum(
        out,
        [
            "Остаток факт в валюте",
            "Начисленные проценты факт в валюте",
            "Просроченные проценты факт в валюте",
        ],
    )

    base_key = composite_key(base, BASE_COLS, ENTITY_KEY_FIELDS)
    monthly_key = composite_key(monthly, MONTHLY_COLS, ENTITY_KEY_FIELDS)
    monthly_date = parse_mixed_dates(monthly[MONTHLY_COLS["report_date"]]).dt.normalize()
    ensure_unique_key_date(monthly, monthly_key, monthly_date, "monthly")

    # Reserve key/date validation.
    reserve_key = reserve_date = None
    if reserve is not None:
        try:
            require_mapped_columns(
                reserve,
                RESERVE_COLS,
                RESERVE_KEY_FIELDS + ["report_date", "reserve"],
                "reserve",
            )
            reserve_key = composite_key(reserve, RESERVE_COLS, RESERVE_KEY_FIELDS)
            reserve_date = parse_mixed_dates(reserve[RESERVE_COLS["report_date"]]).dt.normalize()
            ensure_unique_key_date(reserve, reserve_key, reserve_date, "reserve")
        except Exception:
            raise

    currency = out["Код валюты"]

    # -------------------------------------------------------------------------
    # 10.2. Помесячные исходные показатели
    # -------------------------------------------------------------------------
    for date in report_dates:
        date_ts = pd.to_datetime(date, format=DATE_FMT)
        mask = monthly_date.eq(date_ts)
        part = monthly.loc[mask].copy()
        part_key = monthly_key.loc[mask]

        def pull(logical: str) -> pd.Series:
            real_col = MONTHLY_COLS.get(logical)
            if not real_col or real_col not in part.columns:
                diagnostics.add(
                    "Нет monthly колонки",
                    f"{logical!r} / {real_col!r} отсутствует для {date}; будет NaN",
                    Дата=date,
                )
                return pd.Series(np.nan, index=out.index)
            mapping = dict(zip(part_key, part[real_col]))
            return base_key.map(mapping)

        od_ccy = pull("od_ccy")
        accrued_ccy = pull("accrued_ccy")
        overdue_ccy = pull("overdue_ccy")

        out[pcol("ОД в валюте", date)] = od_ccy.values
        out[pcol("Начисленные проценты в валюте", date)] = accrued_ccy.values
        out[pcol("Просроченные проценты в валюте", date)] = overdue_ccy.values
        out[pcol("ОД+проценты в валюте", date)] = numeric_sum(
            out,
            [
                pcol("ОД в валюте", date),
                pcol("Начисленные проценты в валюте", date),
                pcol("Просроченные проценты в валюте", date),
            ],
        )

        direct_od_byn = pull("od_byn")
        direct_accrued_byn = pull("accrued_byn")
        direct_overdue_byn = pull("overdue_byn")

        fx_map = get_fx_map(fx, date, diagnostics)
        fx_rate = currency.map(normalize_scalar).map(fx_map)

        def resolve_byn(ccy_values: pd.Series, direct_byn: pd.Series) -> pd.Series:
            calculated = to_num(ccy_values) * to_num(fx_rate)
            direct = to_num(direct_byn)
            if PREFER_DIRECT_BYN:
                return direct.where(direct.notna(), calculated)
            return calculated.where(calculated.notna(), direct)

        out[pcol("ОД в BYN", date)] = resolve_byn(od_ccy, direct_od_byn).values
        out[pcol("Начисленные проценты в BYN", date)] = resolve_byn(
            accrued_ccy, direct_accrued_byn
        ).values
        out[pcol("Просроченные проценты в BYN", date)] = resolve_byn(
            overdue_ccy, direct_overdue_byn
        ).values
        out[pcol("ОД+проценты в BYN", date)] = numeric_sum(
            out,
            [
                pcol("ОД в BYN", date),
                pcol("Начисленные проценты в BYN", date),
                pcol("Просроченные проценты в BYN", date),
            ],
        )

        out[pcol("Рейтинг", date)] = pull("rating").values
        out[pcol("Изменение рейтинга", date)] = calculate_rating_change(
            out[pcol("Рейтинг", date)],
            out["Рейтинг на дату договора"],
        ).values
        out[pcol("Зона проблемности", date)] = pull("problem_zone").values
        out[pcol("Стадия", date)] = pull("stage").values

        # Reserve lookup.
        if reserve is not None and reserve_key is not None and reserve_date is not None:
            rmask = reserve_date.eq(date_ts)
            rpart = reserve.loc[rmask]
            rkey = reserve_key.loc[rmask]
            rmap = dict(zip(rkey, rpart[RESERVE_COLS["reserve"]]))

            # Ключи base и reserve могут состоять из одинаковых логических полей.
            if RESERVE_KEY_FIELDS == ENTITY_KEY_FIELDS:
                left_rkey = base_key
            else:
                left_rkey = composite_key(base, BASE_COLS, RESERVE_KEY_FIELDS)

            out[pcol("Резерв по МСФО", date)] = left_rkey.map(rmap).values
        else:
            out[pcol("Резерв по МСФО", date)] = np.nan

        out[pcol("%резервирования МСФО", date)] = calculate_msfo_rate(
            out[pcol("ОД+проценты в BYN", date)],
            out[pcol("Резерв по МСФО", date)],
        ).values

    # -------------------------------------------------------------------------
    # 10.3. Факторный анализ между соседними датами
    # -------------------------------------------------------------------------
    first = report_dates[0]
    out[pcol("Ухудшение качества", first)] = 0.0
    out[pcol("Переоценка", first)] = 0.0
    out[pcol("Движение портфеля", first)] = 0.0

    for prev_date, curr_date in zip(report_dates[:-1], report_dates[1:]):
        prev_rate = out[pcol("%резервирования МСФО", prev_date)]
        curr_rate = out[pcol("%резервирования МСФО", curr_date)]

        quality = calculate_quality_effect(
            out[pcol("ОД+проценты в BYN", curr_date)],
            prev_rate,
            curr_rate,
        )

        revaluation = calculate_revaluation_effect(
            out[pcol("ОД+проценты в валюте", prev_date)],
            currency,
            prev_date,
            curr_date,
            prev_rate,
            fx,
            diagnostics,
        )

        movement = calculate_portfolio_movement(
            out[pcol("ОД+проценты в BYN", prev_date)],
            out[pcol("ОД+проценты в BYN", curr_date)],
            prev_rate,
            revaluation,
        )

        out[pcol("Ухудшение качества", curr_date)] = quality.values
        out[pcol("Переоценка", curr_date)] = revaluation.values
        out[pcol("Движение портфеля", curr_date)] = movement.values

    # -------------------------------------------------------------------------
    # 10.4. Финальный порядок колонок
    # -------------------------------------------------------------------------
    static_order = [name for _, name in MSFO_STATIC_FIELDS]
    # Две рассчитанные фактические суммы ставим рядом с фактическими компонентами.
    insert_after_byn = static_order.index("Просроченные проценты факт в BYN") + 1
    static_order.insert(insert_after_byn, "ОД+проценты факт в BYN")
    insert_after_ccy = static_order.index("Просроченные проценты факт в валюте") + 1
    static_order.insert(insert_after_ccy, "ОД+проценты факт в валюте")

    period_order = [pcol(base_name, d) for d in report_dates for base_name in MSFO_PERIOD_BASES]
    out = out[static_order + period_order].reset_index(drop=True)

    return out, report_dates, diagnostics


## 11. Формулы `df_out_rez`

In [12]:
def contract_term_years(start_date: pd.Series, end_date: pd.Series) -> pd.Series:
    start = parse_mixed_dates(start_date)
    end = parse_mixed_dates(end_date)
    return ((end - start).dt.days + 1) / 365.0


def remaining_term_years(
    end_date: pd.Series,
    report_date: str,
    contract_term: pd.Series,
) -> pd.Series:
    end = parse_mixed_dates(end_date)
    report = pd.to_datetime(report_date, format=DATE_FMT)
    raw = ((end - report).dt.days + 1) / 365.0
    # По постановке: если > 0 — берем остаточный срок, иначе полный срок договора.
    return raw.where(raw > 0, contract_term)


def make_subportfolio_code(segment_2027: pd.Series, rating: pd.Series) -> pd.Series:
    segment_small = (
        segment_2027.astype("string")
        .str.strip()
        .str.lower()
        .eq("малый")
    )

    rating_text = rating.map(lambda x: "" if pd.isna(x) else str(x).strip().removesuffix(".0"))
    prefix = pd.Series(np.where(segment_small, "10_", "3_"), index=segment_2027.index)
    return prefix + rating_text


def normalize_default_date(series: pd.Series) -> pd.Series:
    def one(x):
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return 0
        text = str(x).strip().lower()
        if text in {"", "0", "0.0", "nan", "none", "nat"}:
            return 0
        ts = parse_mixed_date(x)
        return 0 if pd.isna(ts) else ts
    return series.map(one)


def default_age_days(default_date: pd.Series, report_date: str) -> pd.Series:
    report = pd.to_datetime(report_date, format=DATE_FMT)

    def one(x):
        if x == 0 or pd.isna(x):
            return 0
        return (report - pd.Timestamp(x)).days + 1

    return default_date.map(one)


# -----------------------------------------------------------------------------
# TODO-ФОРМУЛЫ: пользователь даст позже.
# Менять потом нужно ТОЛЬКО тело этих функций.
# -----------------------------------------------------------------------------

def calculate_lgd_id(df_rez: pd.DataFrame, date: str) -> pd.Series:
    """TODO: вставить будущую формулу LGD ID."""
    return pd.Series(np.nan, index=df_rez.index)


def calculate_rez_msfo_reserve(df_rez: pd.DataFrame, date: str) -> pd.Series:
    """TODO: вставить будущую формулу резерва по МСФО для df_out_rez."""
    return pd.Series(np.nan, index=df_rez.index)


def calculate_rez_msfo_rate(df_rez: pd.DataFrame, date: str) -> pd.Series:
    """TODO: вставить будущую формулу %рез по МСФО для df_out_rez."""
    return pd.Series(np.nan, index=df_rez.index)


## 12. Построение `df_out_rez`

In [13]:
def build_df_out_rez(
    df_out_msfo: pd.DataFrame,
    report_dates: Sequence[str],
    diagnostics: Optional[Diagnostics] = None,
) -> Tuple[pd.DataFrame, Diagnostics]:
    diagnostics = diagnostics or Diagnostics()

    required = [
        "Баланс/внебаланс/списанные",
        "Инд оценка МСФО",
        "Отчетная дата баланса",
        "Номер договора",
        "ОД+проценты факт в BYN",
        "Стадия факт",
        "%рез факт по МСФО",
        "Дата начала договора",
        "Дата окончания договора",
        "Сегмент 2027",
        "Дата дефолта",
    ]
    missing = [c for c in required if c not in df_out_msfo.columns]
    if missing:
        raise KeyError(f"df_out_msfo: отсутствуют обязательные для df_out_rez колонки: {missing}")

    rez = pd.DataFrame(index=df_out_msfo.index.copy())

    rez["Баланс/внебаланс/списанные"] = df_out_msfo["Баланс/внебаланс/списанные"]
    rez["Инд"] = df_out_msfo["Инд оценка МСФО"]
    rez["Отчетная дата баланса"] = df_out_msfo["Отчетная дата баланса"]
    rez["Номер договора"] = df_out_msfo["Номер договора"]
    rez["ОД+проценты факт в BYN"] = df_out_msfo["ОД+проценты факт в BYN"]
    rez["Стадия факт"] = df_out_msfo["Стадия факт"]
    rez["%рез факт по МСФО"] = df_out_msfo["%рез факт по МСФО"]

    term = contract_term_years(
        df_out_msfo["Дата начала договора"],
        df_out_msfo["Дата окончания договора"],
    )
    default_date = normalize_default_date(df_out_msfo["Дата дефолта"])

    # Дополнительные статические поля временно доступны внутри расчетов через shadow DataFrame.
    shadow = pd.concat([rez, df_out_msfo], axis=1)
    shadow = shadow.loc[:, ~shadow.columns.duplicated(keep="first")]

    for date in report_dates:
        # ---------------------------------------------------------------------
        # Прямое копирование из df_out_msfo
        # ---------------------------------------------------------------------
        rez[pcol("ОД+проценты в BYN", date)] = df_out_msfo[pcol("ОД+проценты в BYN", date)].values
        rez[pcol("Стадия", date)] = df_out_msfo[pcol("Стадия", date)].values
        rez[pcol("Срок договора", date)] = term.values
        rez[pcol("Инд", date)] = rez["Инд"].values

        remaining = remaining_term_years(
            df_out_msfo["Дата окончания договора"],
            date,
            term,
        )
        rez[pcol("Срок до погашения", date)] = remaining.values

        subportfolio = make_subportfolio_code(
            df_out_msfo["Сегмент 2027"],
            df_out_msfo[pcol("Рейтинг", date)],
        )
        rez[pcol("Субпортфель код", date)] = subportfolio.values
        rez[pcol("Ключ субпортфель_рейтинг", date)] = subportfolio.values

        # Сроки для PDLT.
        remaining_num = to_num(rez[pcol("Срок до погашения", date)])
        rez[pcol("Срок для PDLT max", date)] = np.ceil(remaining_num).astype("Int64")
        rez[pcol("Срок для PDLT min", date)] = np.floor(remaining_num).astype("Int64")

        # Чтобы generic lookup видел уже рассчитанные period columns, обновляем shadow.
        for c in [
            pcol("Субпортфель код", date),
            pcol("Ключ субпортфель_рейтинг", date),
            pcol("Срок для PDLT max", date),
            pcol("Срок для PDLT min", date),
            pcol("Стадия", date),
        ]:
            shadow[c] = rez[c]

        # ---------------------------------------------------------------------
        # CCF / macro / PD PIT base
        # ---------------------------------------------------------------------
        rez[pcol("CCF", date)] = lookup_series(
            shadow, date, LOOKUP_SPECS["ccf"], diagnostics, "CCF"
        ).values

        rez[pcol("Макронадбавка", date)] = lookup_series(
            shadow, date, LOOKUP_SPECS["macro"], diagnostics, "Макронадбавка"
        ).values

        base_pd_pit = to_num(
            lookup_series(shadow, date, LOOKUP_SPECS["pd_pit"], diagnostics, "PD PIT")
        )

        # По постановке:
        # если срок > 1 -> lookup как есть;
        # иначе lookup * срок до погашения.
        pd_pit = base_pd_pit.where(remaining_num > 1, base_pd_pit * remaining_num)
        rez[pcol("PD PIT", date)] = pd_pit.values

        shadow[pcol("PD PIT", date)] = rez[pcol("PD PIT", date)]

        # ---------------------------------------------------------------------
        # PD min / max и интерполяция PDLT
        # ---------------------------------------------------------------------
        pd_max = to_num(
            lookup_series(shadow, date, LOOKUP_SPECS["pdlt_max"], diagnostics, "PD max")
        )
        pd_min = to_num(
            lookup_series(shadow, date, LOOKUP_SPECS["pdlt_min"], diagnostics, "PD min")
        )

        rez[pcol("PD max", date)] = pd_max.values
        rez[pcol("PD min", date)] = pd_min.values

        floor_term = np.floor(remaining_num)
        interpolation = pd_min + (pd_max - pd_min) * (remaining_num - floor_term)

        pdlt_gt_year = interpolation.where(remaining_num > 1, pd_pit)
        pdlt = interpolation.where(remaining_num >= 1, pd_pit)

        rez[pcol("PDLT_больше года", date)] = pdlt_gt_year.values
        rez[pcol("PDLT", date)] = pdlt.values

        # ---------------------------------------------------------------------
        # Default
        # ---------------------------------------------------------------------
        rez[pcol("Дата в дефолте", date)] = default_date.values
        rez[pcol("Срок дефолта", date)] = default_age_days(default_date, date).values

        # ---------------------------------------------------------------------
        # Формулы, которые будут даны позже
        # ---------------------------------------------------------------------
        rez[pcol("LGD ID", date)] = calculate_lgd_id(rez, date).values
        rez[pcol("Резерв по МСФО", date)] = calculate_rez_msfo_reserve(rez, date).values
        rez[pcol("%рез по МСФО", date)] = calculate_rez_msfo_rate(rez, date).values

        # Обновим shadow на случай будущих lookup/формул.
        for base_name in REZ_PERIOD_BASES:
            col = pcol(base_name, date)
            if col in rez.columns:
                shadow[col] = rez[col]

    period_order = [pcol(base_name, d) for d in report_dates for base_name in REZ_PERIOD_BASES]
    rez = rez[REZ_STATIC_FIELDS + period_order].reset_index(drop=True)

    return rez, diagnostics


## 13. Единый запуск

После того как ты создал входные DataFrame и поправил конфигурацию, достаточно выполнить:

```python
df_out_msfo, df_out_rez, df_msfo_issues = run_pipeline()
```


In [14]:
def run_pipeline():
    base = get_named_df(INPUT_DF_NAMES["base"], required=True)
    monthly = get_named_df(INPUT_DF_NAMES["monthly"], required=True)
    reserve = get_named_df(INPUT_DF_NAMES["reserve"], required=False)
    fx = get_named_df(INPUT_DF_NAMES["fx"], required=False)

    diagnostics = Diagnostics()

    df_out_msfo, report_dates, diagnostics = build_df_out_msfo(
        base=base,
        monthly=monthly,
        reserve=reserve,
        fx=fx,
        diagnostics=diagnostics,
    )

    df_out_rez, diagnostics = build_df_out_rez(
        df_out_msfo=df_out_msfo,
        report_dates=report_dates,
        diagnostics=diagnostics,
    )

    df_msfo_issues = diagnostics.to_dataframe()

    print(f"df_out_msfo: {df_out_msfo.shape}")
    print(f"df_out_rez:  {df_out_rez.shape}")
    print(f"Отчетные даты: {report_dates}")
    print(f"Диагностических записей: {len(df_msfo_issues)}")

    return df_out_msfo, df_out_rez, df_msfo_issues


# Когда реальные входные DF готовы, выполни:
# df_out_msfo, df_out_rez, df_msfo_issues = run_pipeline()


## 14. Контроль структуры

In [15]:
def describe_output_structure(report_dates: Sequence[str]) -> None:
    print("df_out_msfo — статических колонок:", len(MSFO_STATIC_FIELDS) + 2)
    print("df_out_msfo — колонок на отчетную дату:", len(MSFO_PERIOD_BASES))
    print("df_out_rez — статических колонок:", len(REZ_STATIC_FIELDS))
    print("df_out_rez — колонок на отчетную дату:", len(REZ_PERIOD_BASES))
    print("Количество дат:", len(report_dates))
    print()
    print("Пример блока df_out_msfo:")
    if report_dates:
        for x in MSFO_PERIOD_BASES:
            print("  ", pcol(x, report_dates[0]))
    print()
    print("Пример блока df_out_rez:")
    if report_dates:
        for x in REZ_PERIOD_BASES:
            print("  ", pcol(x, report_dates[0]))


# 15. Демонстрационный расчет

Ниже полностью автономный пример на одном договоре. Он **не использует твои реальные данные** и нужен только для проверки механики.

### Условие примера

- сегмент 2027 = `малый`;
- рейтинг на отчетную дату = `7` → субпортфель `10_7`;
- срок до погашения между 1 и 2 годами;
- `PD min = 3%`, `PD max = 5.5%`;
- PDLT считается линейной интерполяцией между целыми сроками;
- для факторного анализа показан переход между двумя месяцами.


In [16]:
# Независимый пример формул без подключения внешних таблиц.

example_report_date = pd.Timestamp("2026-09-01")
example_end_date = pd.Timestamp("2027-10-15")

example_remaining = (
    (example_end_date - example_report_date).days + 1
) / 365

example_rating = 7
example_segment_2027 = "малый"
example_subportfolio = f"10_{example_rating}" if example_segment_2027.lower() == "малый" else f"3_{example_rating}"

example_pd_pit_base = 0.03
example_pd_min = 0.03
example_pd_max = 0.055
example_floor = math.floor(example_remaining)
example_ceil = math.ceil(example_remaining)

example_pd_pit = (
    example_pd_pit_base
    if example_remaining > 1
    else example_pd_pit_base * example_remaining
)

example_pdlt = (
    example_pd_pit
    if example_remaining < 1
    else example_pd_min
         + (example_pd_max - example_pd_min)
         * (example_remaining - example_floor)
)

# Пример факторного анализа МСФО.
prev_exposure_byn = 9_500_000
curr_exposure_byn = 10_000_000
prev_exposure_ccy = 3_000_000
prev_fx = 3.20
curr_fx = 3.30
prev_rate = 8.0
curr_rate = 10.0

quality = -curr_exposure_byn * (curr_rate - prev_rate) / 100
revaluation = prev_exposure_ccy * (curr_fx - prev_fx) * prev_rate / 100
movement = -(
    curr_exposure_byn * prev_rate / 100
    - prev_exposure_byn * prev_rate / 100
) - revaluation

example = pd.DataFrame({
    "Показатель": [
        "Срок до погашения, лет",
        "Субпортфель код",
        "Срок PDLT min",
        "Срок PDLT max",
        "PD PIT",
        "PDLT",
        "Ухудшение качества, BYN",
        "Переоценка, BYN",
        "Движение портфеля, BYN",
    ],
    "Значение": [
        example_remaining,
        example_subportfolio,
        example_floor,
        example_ceil,
        example_pd_pit,
        example_pdlt,
        quality,
        revaluation,
        movement,
    ],
})

display(example)


,Показатель,Значение
0,"Срок до погашения, лет",1.123288
1,Субпортфель код,10_7
2,Срок PDLT min,1
3,Срок PDLT max,2
4,PD PIT,0.03
5,PDLT,0.033082
6,"Ухудшение качества, BYN",-200000.0
7,"Переоценка, BYN",24000.0
8,"Движение портфеля, BYN",-64000.0


## 16. Как читать пример

При отчетной дате `01.09.2026` и погашении `15.10.2027` остаточный срок получается чуть больше года. Поэтому:

```text
Сегмент 2027 = малый
Рейтинг = 7
=> Субпортфель код = 10_7
```

Если таблица PDLT дает:

```text
10_7, срок 1 -> PD = 0.030
10_7, срок 2 -> PD = 0.055
```

то для дробного срока между 1 и 2 годами используется линейная интерполяция:

```text
PDLT = PD_min + (PD_max - PD_min) * (Срок до погашения - floor(Срок до погашения))
```

Для факторного анализа в примере:

```text
экспозиция: 9.5 млн BYN -> 10.0 млн BYN
%рез:       8% -> 10%
курс:       3.20 -> 3.30
```

и отдельно считаются эффект изменения качества, валютная переоценка и движение портфеля.


# 17. Что именно нужно будет заполнить позже

Перед подключением реальных данных пройди сверху вниз только по этому чек-листу:

1. **`INPUT_DF_NAMES`** — написать имена твоих DataFrame.
2. **`BASE_COLS`, `MONTHLY_COLS`, `RESERVE_COLS`** — поставить реальные названия колонок.
3. **`ENTITY_KEY_FIELDS`** — проверить, достаточно ли `УНП + договор + валюта` для уникальной строки. Если нет — добавить `balance_type` или другой ключ.
4. **`REPORT_DATES`** — оставить `None` для автоопределения либо задать прогнозные даты вручную.
5. **`MSFO_RATE_MODE`** — подтвердить формулу `%резервирования МСФО`.
6. **`LOOKUP_SPECS['ccf']`** — указать ключи и значение для CCF.
7. **`LOOKUP_SPECS['macro']`** — указать ключи макронадбавки.
8. **`LOOKUP_SPECS['pd_pit']`** — указать таблицу PD PIT.
9. **`LOOKUP_SPECS['pdlt_min/max']`** — указать таблицу term-structure PD.
10. Когда будут известны правила — заменить только три функции:
    - `calculate_lgd_id()`;
    - `calculate_rez_msfo_reserve()`;
    - `calculate_rez_msfo_rate()`.

После этого запуск остается один:

```python
df_out_msfo, df_out_rez, df_msfo_issues = run_pipeline()
```
